# Introduction to Causal Inference: DoWhy and the Lalonde Dataset

Does a job training program actually cause participants to earn more, or do people who enroll in training simply differ from those who do not? This is the central challenge of **causal inference**: distinguishing genuine treatment effects from confounding differences between groups. A simple comparison of average earnings between participants and non-participants can be misleading if the two groups differ in age, education, or prior employment history.

**DoWhy** is a Python library that provides a principled, end-to-end framework for causal inference. It organizes the analysis into four explicit steps --- **Model, Identify, Estimate, Refute** --- each of which forces the analyst to state and test causal assumptions rather than hiding them inside a black-box estimator. In this tutorial, we apply DoWhy to the **Lalonde dataset**, a classic dataset from the National Supported Work (NSW) Demonstration program, to estimate how much the job training program increased participants' earnings in 1978.

**Learning objectives:**

- Understand DoWhy's four-step causal inference workflow (Model, Identify, Estimate, Refute)
- Define a causal graph that encodes domain knowledge about confounders
- Identify causal estimands from the graph using the backdoor criterion
- Estimate causal effects using multiple methods (regression adjustment, IPW, doubly robust, propensity score stratification, propensity score matching)
- Assess robustness of estimates using refutation tests

## DoWhy's four-step framework

Most statistical software lets you jump straight from data to estimates, skipping the hard work of stating assumptions and testing whether the results are trustworthy. DoWhy takes a different approach: it organizes every causal analysis into four explicit steps, each answering a distinct question.

**The four steps:**

1. **Model** --- *"What are the causal relationships?"* Encode your domain knowledge as a causal graph (a DAG). This is where you declare which variables cause which, making your assumptions explicit and debatable rather than hidden inside a regression.
2. **Identify** --- *"Can we estimate the effect from data?"* Given the graph, DoWhy uses graph theory to determine whether the causal effect is identifiable --- meaning it can be computed from observed data alone --- and returns the mathematical formula (the *estimand*) needed to do so.
3. **Estimate** --- *"What is the causal effect?"* Apply one or more statistical methods to compute the actual numeric estimate. DoWhy supports multiple estimators so you can check whether different methods agree.
4. **Refute** --- *"Should we trust the estimate?"* Run automated falsification tests that probe whether the result could be a statistical artifact, whether it is sensitive to unobserved confounders, and whether it is stable across subsamples.

The ordering is deliberate. You cannot estimate a causal effect without first identifying the correct formula, and you cannot identify the formula without first specifying your causal assumptions. This sequential discipline is DoWhy's key contribution: it prevents the common mistake of running a regression and calling the coefficient "causal" without ever checking whether the adjustment set is correct or whether the result survives basic robustness checks.

## Setup and imports

Install the required package if needed:

In [ ]:
!pip install dowhy -q

The following code imports all necessary libraries and sets configuration variables. We define the outcome, treatment, and covariate columns that will be used throughout the analysis.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression as SklearnLR
from dowhy import CausalModel
from dowhy.datasets import lalonde_dataset

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configuration
OUTCOME = "re78"
OUTCOME_LABEL = "Earnings in 1978 (USD)"
TREATMENT = "treat"
TREATMENT_LABEL = "Job Training (treat)"
COVARIATES = ["age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75"]

# Site color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"

## Data loading: The Lalonde Dataset

The Lalonde dataset comes from the **National Supported Work (NSW) Demonstration**, a randomized employment program conducted in the 1970s in the United States. Eligible applicants --- mostly disadvantaged workers with limited employment histories --- were randomly assigned to receive job training (treatment) or not (control). The dataset records each participant's demographics, prior earnings, and post-program earnings in 1978. It has become a benchmark for testing causal inference methods because the random assignment provides a credible ground truth against which observational estimators can be compared.

DoWhy includes the Lalonde dataset directly, so we can load it with a single function call.

In [ ]:
df = lalonde_dataset()

# Convert boolean treatment to integer for DoWhy compatibility
df[TREATMENT] = df[TREATMENT].astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nTreatment groups:")
print(df[TREATMENT].value_counts().sort_index().rename({0: "Control", 1: "Training"}))
print(f"\nOutcome ({OUTCOME}) summary:")
print(df[OUTCOME].describe().round(2))

The dataset contains 445 participants with 12 variables. The treatment is split into 185 individuals who received job training and 260 controls who did not. The outcome variable, real earnings in 1978 (`re78`), has a mean of \$5,301 but enormous variation (standard deviation of \$6,631), ranging from \$0 to \$60,308. The median (\$3,702) is well below the mean, indicating a right-skewed distribution --- many participants earned little or nothing while a few earned substantially more.

## Exploratory data analysis

### Outcome distribution by treatment group

Before any causal modeling, we compare the raw earnings distributions between training and control groups. If the training program had an effect, we expect to see higher average earnings in the training group --- but we cannot yet tell whether any difference is truly caused by the program or driven by pre-existing differences between the groups.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for group, label, color in [(0, "Control", STEEL_BLUE), (1, "Training", WARM_ORANGE)]:
    subset = df[df[TREATMENT] == group][OUTCOME]
    ax.hist(subset, bins=30, alpha=0.6, label=f"{label} (mean=${subset.mean():,.0f})",
            color=color, edgecolor="white")
ax.set_xlabel(OUTCOME_LABEL)
ax.set_ylabel("Count")
ax.set_title(f"Distribution of {OUTCOME_LABEL} by Treatment Group")
ax.legend()
plt.show()

Both distributions are heavily right-skewed, with a large spike near zero reflecting participants who had no earnings. The training group has a higher mean (\$6,349) compared to the control group (\$4,555), a raw difference of about \$1,794. However, both distributions overlap substantially, and the spike at zero is present in both groups, indicating that many participants struggled to find employment regardless of training.

### Covariate balance

In a randomized experiment, we expect the covariates to be balanced across treatment and control groups. Under randomization, the naive difference-in-means is **unbiased** for the ATE in expectation --- but with a finite sample of 445 observations, chance imbalances can still arise and reduce the precision of the estimate. Checking covariate balance helps us assess whether such imbalances exist and whether covariate adjustment could improve efficiency. We examine categorical and continuous covariates separately, since their scales differ substantially.

#### Categorical covariates

The four binary covariates --- `black`, `hisp`, `married`, and `nodegr` (no high school degree) --- indicate demographic group membership. Comparing their proportions across treatment and control groups reveals whether random assignment produced balanced groups on these characteristics.

In [ ]:
categorical_vars = ["black", "hisp", "married", "nodegr"]
cat_means = df.groupby(TREATMENT)[categorical_vars].mean()

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(categorical_vars))
width = 0.35
ax.bar(x - width / 2, cat_means.loc[0], width, label="Control",
       color=STEEL_BLUE, edgecolor="white")
ax.bar(x + width / 2, cat_means.loc[1], width, label="Training",
       color=WARM_ORANGE, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(categorical_vars, rotation=45, ha="right")
ax.set_ylabel("Proportion")
ax.set_ylim(0, 1)
ax.set_title("Covariate Balance: Categorical Variables")
ax.legend()
plt.show()

The categorical covariates are well balanced across treatment and control groups, consistent with random assignment. The sample is predominantly Black (83%) and has a high rate of lacking a high school diploma (78%), reflecting the disadvantaged population targeted by the NSW program. Hispanic and married proportions are low in both groups (roughly 6% and 16%, respectively), with no meaningful differences between treatment arms.

#### Covariate balance: Standardized Mean Differences

Comparing raw group means can be misleading when covariates are measured on different scales. Suppose the control group earns \$500 more in prior earnings (`re74`) than the training group, and is also 1 year older on average. Which imbalance is larger? The raw numbers cannot answer this question --- \$500 sounds like a lot, but prior earnings vary by thousands of dollars across individuals, so a \$500 gap may be trivial relative to the spread. A 1-year age difference sounds small, but if most participants are clustered around age 25, that gap may represent a meaningful shift in the distribution.

The **Standardized Mean Difference (SMD)** resolves this by asking: *how many standard deviations apart are the treatment and control groups on each covariate?* For each variable, we compute the difference in group means and divide by the pooled standard deviation. This converts every covariate --- whether binary, measured in years, or measured in dollars --- to the same unitless scale, making imbalances directly comparable:

$$\text{SMD} = \frac{\bar{X}_{treated} - \bar{X}_{control}}{\sqrt{(s^2_{treated} + s^2_{control}) \,/\, 2}}$$

An absolute SMD below 0.1 is the conventional threshold for "good balance" (Austin, 2011). Values above 0.1 signal that the groups differ by more than one-tenth of a standard deviation on that variable --- enough to potentially confound the treatment effect estimate. A **Love plot** displays the absolute SMD for all covariates as horizontal bars, with a dashed line at the 0.1 threshold. Bars in steel blue fall below the threshold (balanced), while bars in warm orange exceed it (imbalanced).

In [ ]:
# Standardized Mean Difference (SMD) for all covariates
treated = df[df[TREATMENT] == 1]
control = df[df[TREATMENT] == 0]

smd_values = {}
for var in COVARIATES:
    diff = treated[var].mean() - control[var].mean()
    pooled_sd = np.sqrt((treated[var].std()**2 + control[var].std()**2) / 2)
    smd_values[var] = diff / pooled_sd

smd_df = pd.DataFrame({"variable": list(smd_values.keys()),
                        "smd": list(smd_values.values())})
smd_df["abs_smd"] = smd_df["smd"].abs()
smd_df = smd_df.sort_values("abs_smd")

fig, ax = plt.subplots(figsize=(8, 5))
colors = [STEEL_BLUE if v < 0.1 else WARM_ORANGE for v in smd_df["abs_smd"]]
ax.barh(smd_df["variable"], smd_df["abs_smd"], color=colors,
        edgecolor="white", height=0.6)
ax.axvline(0.1, color=NEAR_BLACK, linewidth=1, linestyle="--", label="SMD = 0.1 threshold")
ax.set_xlabel("Absolute Standardized Mean Difference")
ax.set_title("Covariate Balance: Love Plot (All Covariates)")
ax.legend(loc="lower right")
plt.show()

The Love plot reveals a more nuanced picture than raw mean comparisons would suggest. Prior earnings (`re74` and `re75`) --- which appeared imbalanced when comparing raw means in the thousands --- are actually well balanced on the standardized scale (SMD < 0.1), because their large variances absorb the mean differences. In contrast, `nodegr` shows the largest imbalance (SMD ~0.31), followed by `hisp` (~0.18) and `educ` (~0.14). These imbalances, despite random assignment, reflect the small sample size and the disadvantaged population targeted by NSW. Although the naive difference-in-means remains unbiased under randomization, adjusting for these chance imbalances can **improve the precision** of the treatment effect estimate --- a well-known result in the experimental design literature (Lin, 2013; Freedman, 2008).

## The causal inference problem

### ATE vs ATT: Two different causal questions

Before estimating the treatment effect, we need to be precise about *which* causal question we are asking. There are two distinct estimands, each answering a different policy-relevant question:

- **Average Treatment Effect (ATE)** --- *"What would happen if we assigned treatment to a random person from the entire population?"* The ATE averages the treatment effect over **everyone** --- both the treated and the untreated:

$$\text{ATE} = E[Y(1) - Y(0)]$$

- **Average Treatment Effect on the Treated (ATT)** --- *"What was the effect of treatment for those who actually received it?"* The ATT averages the treatment effect only over the **treated** subpopulation:

$$\text{ATT} = E[Y(1) - Y(0) \mid T = 1]$$

The distinction matters because the people who receive treatment may differ systematically from those who do not. If the training program helps disadvantaged workers the most, and disadvantaged workers are more likely to enroll, then the ATT (the effect on those who enrolled) will be larger than the ATE (the effect if we enrolled everyone at random). Conversely, if the program is most effective for workers who are *least* likely to enroll, the ATE could exceed the ATT.

**In this tutorial, we estimate the ATE** --- the average effect of the NSW job training program across the entire study population. This is the natural estimand for a randomized experiment where we want to evaluate the program's overall impact. Four of our five estimation methods (regression adjustment, IPW, AIPW, and propensity score stratification) target the ATE directly. The exception is **propensity score matching**, which discards unmatched control units and therefore shifts the estimand toward the ATT --- we flag this distinction when we discuss the matching results.

### Why simple comparisons can mislead

A naive approach to estimating the treatment effect is to compute the difference in mean outcomes between the training and control groups. This gives us the **Average Treatment Effect (ATE)**:

$$\text{ATE}_{naive} = \bar{Y}_{treated} - \bar{Y}_{control}$$

While this is a natural starting point and is **unbiased in expectation** under randomization, it can be imprecise when finite-sample covariate imbalances exist. Adjusting for covariates that predict the outcome can sharpen the estimate. In observational studies, the problem is more severe --- without adjustment, the naive estimator can be genuinely biased by confounding.

In [ ]:
mean_treated = df[df[TREATMENT] == 1][OUTCOME].mean()
mean_control = df[df[TREATMENT] == 0][OUTCOME].mean()
naive_ate = mean_treated - mean_control

print(f"Mean earnings (Training): ${mean_treated:,.2f}")
print(f"Mean earnings (Control):  ${mean_control:,.2f}")
print(f"Naive ATE (difference):   ${naive_ate:,.2f}")

The naive estimate suggests that training increases earnings by \$1,794 on average. Under randomization, this estimate is unbiased in expectation, but the finite-sample covariate imbalances we observed earlier (particularly in `nodegr`, `hisp`, and `educ`) mean that covariate adjustment can sharpen the estimate and account for chance differences between groups. This is where DoWhy's structured framework helps --- it forces us to explicitly model our causal assumptions, identify the correct estimand, apply rigorous estimation methods, and test whether the results hold up under scrutiny.

The causal structure we assume is:
- Each covariate (age, educ, black, hisp, married, nodegr, re74, re75) affects both treatment assignment and earnings
- Treatment (`treat`) affects the outcome (`re78`)
- No covariate is itself caused by the treatment (pre-treatment variables)

We now create the `CausalModel` in DoWhy, specifying the treatment, outcome, and common causes. The model object stores the data, the causal graph, and metadata that DoWhy will use in subsequent steps to determine the correct adjustment strategy.

In [ ]:
model = CausalModel(
    data=df,
    treatment=TREATMENT,
    outcome=OUTCOME,
    common_causes=COVARIATES,
)
print("CausalModel created successfully.")

DoWhy can visualize the causal graph it constructed using the `view_model()` method, which uses Graphviz to render the DAG automatically from the model's internal graph representation:

In [ ]:
# Visualize the causal graph using DoWhy's built-in method
model.view_model(layout="dot")

from IPython.display import Image, display
display(Image(filename="causal_model.png"))

The DAG makes our assumptions explicit: the eight covariates are common causes that affect both treatment assignment (`treat`) and earnings (`re78`). The arrows encode the direction of causation --- each confounder points to both `treat` and `re78`, and `treat` points to `re78` (the causal effect we want to estimate). By stating these assumptions as a graph, DoWhy can automatically determine which variables need to be adjusted for and which estimation strategies are valid.

## Step 2: Identify --- Find the causal estimand

With the causal graph defined, DoWhy uses graph theory to **identify** the causal estimand --- the mathematical expression that, if computed correctly, equals the true causal effect. This step determines *whether* the effect is identifiable from the data given our assumptions, and *what* variables we need to condition on.

### What does "identification" mean?

In causal inference, **identification** answers a deceptively simple question: *can we compute the causal effect from the data we have, without running a new experiment?* The answer is not always yes. Consider a scenario where an unmeasured variable (say, "motivation") affects both whether someone enrolls in training and how much they earn afterward. No amount of data on age, education, and prior earnings can untangle the causal effect of training from the confounding effect of motivation --- the causal effect is **not identified** without observing motivation.

Identification is the bridge between *causal assumptions* (encoded in the graph) and *statistical computation* (what we can actually calculate from data). If the effect is identified, the identification step produces an **estimand** --- a precise mathematical formula that tells us exactly which conditional expectations or reweightings to compute. If the effect is not identified, no estimation method can produce a credible causal estimate, no matter how sophisticated.

### Identification strategies

DoWhy checks three main strategies, each applicable in different causal structures:

- **Backdoor criterion** --- The most common strategy. It applies when we can observe all confounders between treatment and outcome. By conditioning on these confounders, we "block" all backdoor paths --- non-causal pathways that create spurious associations. In the Lalonde example, conditioning on the eight covariates satisfies the backdoor criterion because they are the only common causes of `treat` and `re78`.
- **Instrumental variables (IV)** --- Useful when some confounders are *unobserved*. An instrument is a variable that affects treatment but has *no direct effect* on the outcome except through the treatment itself. For example, draft lottery numbers have been used as instruments for military service: the lottery affects whether someone serves (treatment) but has no direct effect on later earnings (outcome) except through the service itself. IV estimation requires strong assumptions but can identify causal effects when backdoor adjustment is impossible.
- **Front-door criterion** --- Applies when there is a **mediator** that fully transmits the treatment effect and is itself unconfounded with the outcome. This strategy is rare in practice but theoretically important: it can identify causal effects even in the presence of unmeasured confounders between treatment and outcome, as long as the mediator pathway is clean.

A key advantage of DoWhy is that **it automates the identification step**. Given the causal graph, DoWhy algorithmically checks which strategies are valid and returns the correct estimand. This prevents a common and dangerous mistake in applied work: manually choosing which variables to "control for" without formally checking whether the chosen adjustment set actually satisfies the conditions for causal identification.

In [ ]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

DoWhy identifies the **backdoor estimand** as the primary identification strategy, expressing the causal effect as the derivative of the conditional expectation of earnings with respect to treatment, conditioning on all eight covariates. The critical assumption is **unconfoundedness** --- there are no unmeasured confounders beyond the ones we specified. DoWhy also checks for instrumental variable and front-door estimands but finds none applicable, which is expected given our graph structure.

## Step 3: Estimate --- Compute the causal effect

With the estimand identified, we now apply statistical methods to compute the actual causal effect estimate. DoWhy supports multiple estimation methods, each with different assumptions and properties. We compare five approaches to see how robust the estimate is across methods.

Causal estimation methods fall into **three broad paradigms**, distinguished by what they model:

1. **Outcome modeling** (Regression Adjustment) --- directly models the relationship $E[Y \mid X, T]$ between covariates, treatment, and outcome. Its validity depends on correctly specifying this outcome model.
2. **Treatment modeling** (IPW, PS Stratification, PS Matching) --- models the treatment assignment mechanism $P(T \mid X)$ (the propensity score) and uses it to remove confounding. All three methods rely exclusively on the propensity score --- they differ in *how* they use it (reweighting, grouping, or pairing observations) but none of them model the outcome. Their validity depends on correctly specifying the propensity score model.
3. **Doubly robust** (AIPW) --- the only true hybrid. It explicitly combines an outcome model $E[Y \mid X, T]$ with a propensity score model $P(T \mid X)$, and is consistent if *either* model is correctly specified. This "double protection" is why it is called doubly robust.

**Method taxonomy:**

- **Outcome Modeling** → Regression Adjustment
- **Treatment Modeling** → IPW, PS Stratification, PS Matching
- **Doubly Robust** → AIPW

The key trade-offs across paradigms are:

- **What each paradigm models**: Outcome modeling specifies how covariates relate to earnings ($E[Y \mid X, T]$). Treatment modeling specifies how covariates relate to treatment assignment ($P(T \mid X)$) --- all three PS methods use this same propensity score but differ in how they apply it. Doubly robust specifies both models simultaneously.
- **What each paradigm assumes**: Regression adjustment requires the outcome model to be correctly specified. All three propensity score methods (IPW, stratification, matching) require the propensity score model to be correctly specified. Doubly robust only requires *one* of the two to be correct.
- **Bias-variance characteristics**: Regression adjustment tends to be low-variance but can be biased if the outcome-covariate relationship is nonlinear. IPW can have high variance when propensity scores are extreme (near 0 or 1). Stratification and matching use the propensity score more conservatively --- by grouping or pairing rather than directly reweighting --- which can reduce variance relative to IPW. Doubly robust balances both concerns but is more complex to implement.

The three treatment modeling methods differ in *how* they use the propensity score to create balanced comparisons:

- **IPW** reweights every observation by the inverse of its propensity score, creating a pseudo-population where treatment is independent of covariates. It uses the full sample but can be unstable when propensity scores are near 0 or 1.
- **PS Stratification** divides observations into groups (strata) with similar propensity scores, then computes simple mean differences within each stratum. By comparing treated and control units within the same stratum, it approximates a block-randomized experiment.
- **PS Matching** pairs each treated unit with the control unit that has the most similar propensity score, then computes mean differences within matched pairs. It discards unmatched observations, focusing on the closest comparisons at the cost of reduced sample size.

None of these methods model the outcome --- they all achieve confounding adjustment purely through the propensity score. If outcome modeling and treatment modeling agree, we can be more confident that neither model is badly misspecified.

### Method 1: Regression Adjustment

The simplest approach --- also called **regression adjustment** --- adjusts for confounders by including them as covariates in a linear regression. The treatment effect is the coefficient on treatment after controlling for covariates. This assumes a linear relationship between covariates and the outcome, which may be restrictive but provides a transparent baseline.

In [ ]:
estimate_ra = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True,
)
print(f"Estimated ATE (Regression Adjustment): ${estimate_ra.value:,.2f}")

The regression adjustment estimate is \$1,676, slightly lower than the naive difference of \$1,794. The reduction from \$1,794 to \$1,676 reflects the covariate adjustment --- by accounting for finite-sample imbalances in age, education, race, and prior earnings, the estimated treatment effect shrinks by about \$118. In this randomized setting, the adjustment primarily improves precision rather than removing bias, but the same technique is essential in observational studies where confounding is a genuine concern.

### Method 2: Inverse Probability Weighting (IPW)

IPW takes a fundamentally different approach from regression adjustment. Instead of modeling the outcome, it models the **treatment assignment mechanism**. Each observation is weighted by the inverse of its probability of receiving the treatment it actually received (the **propensity score**). This reweighting creates a "pseudo-population" in which treatment assignment is independent of the observed confounders.

The IPW estimator is:

$$\hat{\tau}_{IPW} = \frac{1}{n} \sum_{i=1}^{n} \left[ \frac{T_i Y_i}{\hat{e}(X_i)} - \frac{(1 - T_i) Y_i}{1 - \hat{e}(X_i)} \right]$$

where $\hat{e}(X_i)$ is the estimated propensity score for individual $i$.

In [ ]:
estimate_ipw = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    method_params={"weighting_scheme": "ips_weight"},
)
print(f"Estimated ATE (IPW): ${estimate_ipw.value:,.2f}")

The IPW estimate of \$1,559 is the lowest among all methods. IPW is sensitive to extreme propensity scores --- when some individuals have very high or very low probabilities of treatment, their weights become large and can dominate the estimate. The difference from the regression adjustment (\$1,676 vs \$1,559) reflects the fact that IPW makes no assumptions about the outcome model, relying entirely on correct specification of the propensity score model.

### Method 3: Doubly Robust (AIPW)

The **doubly robust** estimator --- also called **Augmented Inverse Probability Weighting (AIPW)** --- combines both regression adjustment and IPW into a single estimator. The key advantage is that the estimate is consistent if *either* the outcome model *or* the propensity score model is correctly specified (hence "doubly robust").

The AIPW estimator is:

$$\hat{\tau}_{DR} = \frac{1}{n} \sum_{i=1}^{n} \left[ \hat{\mu}_1(X_i) - \hat{\mu}_0(X_i) + \frac{T_i (Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1 - T_i)(Y_i - \hat{\mu}_0(X_i))}{1 - \hat{e}(X_i)} \right]$$

where $\hat{\mu}_1(X_i)$ and $\hat{\mu}_0(X_i)$ are the predicted outcomes under treatment and control, and $\hat{e}(X_i)$ is the propensity score.

We implement the AIPW estimator manually rather than using DoWhy's built-in `backdoor.doubly_robust` method, which has a known compatibility issue with recent scikit-learn versions. The manual implementation also makes the estimator's two-component structure --- propensity score model plus outcome model --- fully transparent.

In [ ]:
# Doubly Robust (AIPW) — manual implementation
ps_model = LogisticRegression(max_iter=1000, random_state=42)
ps_model.fit(df[COVARIATES], df[TREATMENT])
ps = ps_model.predict_proba(df[COVARIATES])[:, 1]

outcome_model_1 = SklearnLR().fit(df[df[TREATMENT] == 1][COVARIATES], df[df[TREATMENT] == 1][OUTCOME])
outcome_model_0 = SklearnLR().fit(df[df[TREATMENT] == 0][COVARIATES], df[df[TREATMENT] == 0][OUTCOME])

mu1 = outcome_model_1.predict(df[COVARIATES])
mu0 = outcome_model_0.predict(df[COVARIATES])
T = df[TREATMENT].values
Y = df[OUTCOME].values

dr_ate = np.mean(
    (mu1 - mu0)
    + T * (Y - mu1) / ps
    - (1 - T) * (Y - mu0) / (1 - ps)
)
print(f"Estimated ATE (Doubly Robust): ${dr_ate:,.2f}")

The doubly robust estimate of \$1,620 falls between the regression adjustment (\$1,676) and IPW (\$1,559) estimates. This reflects how the AIPW estimator works: it uses the outcome model as its primary estimate and adds an IPW-weighted correction based on the prediction residuals. In practice, the doubly robust estimator is often the preferred choice because it provides insurance against misspecification of either component model.

### Method 4: Propensity Score Stratification

Propensity score stratification works by first estimating each individual's probability of receiving treatment given their covariates (the **propensity score**), then dividing the sample into strata with similar propensity scores. Within each stratum, treated and control individuals are more comparable, so the within-stratum treatment effect is less biased. The overall ATE is a weighted average of the stratum-specific effects.

In [ ]:
estimate_ps_strat = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_stratification",
    method_params={"num_strata": 5, "clipping_threshold": 5},
)
print(f"Estimated ATE (PS Stratification): ${estimate_ps_strat.value:,.2f}")

Propensity score stratification with 5 strata estimates the ATE at \$1,617, very close to the doubly robust estimate (\$1,620). The stratification approach is more flexible than linear regression because it does not impose a functional form on the outcome-covariate relationship. The estimate is in the same ballpark as the regression result, which is reassuring --- multiple methods agree that the training effect is in the \$1,550--\$1,700 range.

### Method 5: Propensity Score Matching

Propensity score matching pairs each treated individual with one or more control individuals who have similar propensity scores. The treatment effect is estimated by comparing outcomes within matched pairs. This is conceptually the most intuitive approach --- it mimics what we would see if we could directly compare individuals who are identical except for their treatment status.

In [ ]:
estimate_ps_match = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
)
print(f"Estimated ATE (PS Matching): ${estimate_ps_match.value:,.2f}")

Propensity score matching estimates the effect at \$1,736, the highest of the five adjusted estimates and closest to the naive difference. Matching tends to give slightly different results because it uses only the closest comparisons rather than the full sample. As noted above, this estimate is closer to the **ATT** than the ATE, so it answers a slightly different question than the other four methods --- readers should keep this distinction in mind when comparing across estimators. The fact that all five methods produce estimates between \$1,559 and \$1,736 provides strong evidence that the treatment effect is real and robust to the choice of estimation method.

## Step 4: Refute --- Test robustness

The final and perhaps most valuable step in DoWhy's framework is **refutation** --- systematically testing whether the estimated causal effect is robust to violations of our assumptions. DoWhy provides several built-in refutation tests, each probing a different potential weakness.

### Why refutation matters

Most causal inference workflows stop after estimation: you run a regression, get a coefficient, and report it as the causal effect. DoWhy's refutation step is its key innovation --- it provides **automated falsification tests** that probe whether the estimate could be an artifact of the model, the data, or violated assumptions. This is the causal inference equivalent of "stress testing": if the estimate survives multiple attempts to break it, we can be more confident that it reflects a genuine causal relationship.

DoWhy's refutation tests fall into three categories, each targeting a different potential weakness:

- **Placebo tests** --- *"If the treatment doesn't matter, does the effect disappear?"* These tests replace the real treatment with a fake (randomly permuted) treatment. If the estimated effect drops to near zero, the original result is tied to the actual treatment rather than being a statistical artifact of the model or data structure.
- **Sensitivity tests** --- *"If we missed a confounder, does the estimate change?"* These tests add a randomly generated variable as an additional confounder. If the estimate barely changes, it suggests the result is not fragile --- adding one more covariate does not destabilize it.
- **Stability tests** --- *"If we use different data, does the estimate hold?"* These tests re-estimate the effect on random subsets of the data. If the estimate fluctuates wildly, it may depend on a few influential observations rather than reflecting a stable population-level effect.

An important caveat: **passing all refutation tests does not prove causation**. The tests can only detect certain types of problems --- they cannot rule out every possible source of bias. However, **failing any test is a strong signal that something is wrong** and warrants further investigation before drawing causal conclusions.

### Placebo Treatment Test

The placebo test replaces the actual treatment with a randomly permuted version. If our estimate is truly capturing a causal effect, this fake treatment should produce an effect near zero.

In [ ]:
refute_placebo = model.refute_estimate(
    identified_estimand,
    estimate_ra,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=100,
)
print(refute_placebo)

The placebo treatment test produces a new effect close to zero, dramatically smaller than the original estimate of \$1,676. The high p-value indicates that the original estimate is well above what we would expect from a random treatment assignment. This is strong evidence that the estimated effect is not an artifact of the model or data structure.

### Random Common Cause Test

This test adds a randomly generated confounder to the model and checks whether the estimate changes. If our model is correctly specified, adding a random variable should not significantly alter the result.

In [ ]:
refute_random = model.refute_estimate(
    identified_estimand,
    estimate_ra,
    method_name="random_common_cause",
    num_simulations=100,
)
print(refute_random)

Adding a random common cause barely changes the estimate --- the new effect is nearly identical to the original \$1,676. This confirms that the model is not overly sensitive to the specific set of confounders included.

### Data Subset Test

The data subset test re-estimates the effect on random 80% subsamples of the data. If the estimate is robust, it should remain similar across different subsets.

In [ ]:
refute_subset = model.refute_estimate(
    identified_estimand,
    estimate_ra,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
    num_simulations=100,
)
print(refute_subset)

The data subset refuter produces a mean effect close to the full-sample estimate of \$1,676, indicating that the estimate is stable across subsets and does not depend on a handful of outlier observations.

## Comparing all estimates

To visualize how all estimation approaches compare, we plot the ATE estimates side by side. Consistent estimates across different methods strengthen confidence in the causal conclusion.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
methods = ["Naive\n(Diff. in Means)", "Regression\nAdjustment", "IPW",
           "Doubly Robust\n(AIPW)", "PS\nStratification", "PS\nMatching"]
estimates = [naive_ate, estimate_ra.value, estimate_ipw.value,
             dr_ate, estimate_ps_strat.value, estimate_ps_match.value]
colors = ["#999999", STEEL_BLUE, WARM_ORANGE, TEAL, "#e8956a", "#c4623d"]

bars = ax.barh(methods, estimates, color=colors, edgecolor="white", height=0.6)
for bar, val in zip(bars, estimates):
    offset = 50 if val >= 0 else -50
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"${val:,.0f}", va="center", ha=ha, fontsize=10, color=NEAR_BLACK)

ax.axvline(0, color="black", linewidth=0.5, linestyle="--")ax.set_xlabel("Estimated Average Treatment Effect (USD)")ax.set_title("Causal Effect Estimates: NSW Job Training on 1978 Earnings")plt.show()

All six methods produce positive estimates between \$1,559 and \$1,794, indicating that the NSW job training program increased participants' 1978 earnings by roughly \$1,550--\$1,800. The adjusted methods cluster between \$1,559 and \$1,736, suggesting that about \$40--\$235 of the naive estimate was due to finite-sample covariate imbalances rather than the treatment. Notably, the five adjusted estimators represent three distinct paradigms --- outcome modeling (regression adjustment), treatment modeling (IPW, PS stratification, PS matching), and doubly robust (AIPW) --- yet all converge on a similar range, reinforcing confidence in the causal conclusion.

## Summary table

In [ ]:
results_df = pd.DataFrame({
    "Method": ["Naive (Diff. in Means)", "Regression Adjustment", "IPW",
               "Doubly Robust (AIPW)", "PS Stratification", "PS Matching"],
    "Estimated ATE (USD)": [f"${naive_ate:,.0f}", f"${estimate_ra.value:,.0f}",
                            f"${estimate_ipw.value:,.0f}", f"${dr_ate:,.0f}",
                            f"${estimate_ps_strat.value:,.0f}", f"${estimate_ps_match.value:,.0f}"],
    "Notes": ["No covariate adjustment", "Models outcome, assumes linearity",
              "Models treatment assignment", "Models both outcome and treatment",
              "5 strata, flexible", "Nearest-neighbor matching"],
})print(results_df.to_string(index=False))

## Discussion

The Lalonde dataset provides a compelling case study for DoWhy's four-step framework. Each step serves a distinct purpose: the **Model** step forces us to articulate our causal assumptions as a graph, the **Identify** step uses graph theory to determine the correct adjustment formula, the **Estimate** step applies statistical methods to compute the effect, and the **Refute** step probes whether the result withstands scrutiny.

The estimated ATE ranges from \$1,559 (IPW) to \$1,736 (PS matching), with the doubly robust estimate at \$1,620. On average, participants who received job training earned roughly \$1,600 more in 1978 than they would have without training. On a base of \$4,555 for the control group, this represents roughly a 34--38% increase in earnings --- a substantial effect for a disadvantaged population with very low baseline earnings.

The five estimators span three estimation paradigms: outcome modeling (regression adjustment), treatment modeling (IPW, PS stratification, PS matching), and doubly robust (AIPW). Their convergence on the \$1,550--\$1,800 range is particularly compelling because each paradigm relies on different modeling assumptions. All three treatment modeling methods --- IPW, stratification, and matching --- use the propensity score exclusively, differing only in how they apply it (reweighting, grouping, or pairing). None of them model the outcome, which is why they belong to the same paradigm despite their different mechanics.

The key strength of DoWhy over ad-hoc statistical approaches is transparency. The causal graph makes assumptions visible and debatable. The identification step formally checks whether the effect is estimable. Multiple estimation methods let us assess robustness. And refutation tests provide automated sanity checks that would otherwise require expert judgment.

## Takeaways

- **DoWhy's four-step workflow** (Model, Identify, Estimate, Refute) makes causal assumptions explicit and testable, rather than hiding them inside a black-box estimator.
- **The NSW job training program increased 1978 earnings by approximately \$1,550--\$1,800**, a 34--38% gain over the control group mean of \$4,555.
- **Five estimation methods** --- regression adjustment, IPW, doubly robust, PS stratification, and PS matching --- all produce positive, consistent estimates, strengthening confidence in the causal conclusion.
- **The doubly robust (AIPW) estimator** (\$1,620) is the most credible single estimate because it remains consistent if either the outcome model or the propensity score model is misspecified.
- **IPW and regression adjustment represent two complementary paradigms**: modeling treatment assignment (\$1,559) vs. modeling the outcome (\$1,676). Their divergence quantifies sensitivity to modeling choices.
- **Refutation tests confirm robustness** --- the placebo test reduced the effect from \$1,676 to just \$62, ruling out statistical artifacts.
- **Causal graphs encode domain knowledge as testable assumptions**; the backdoor criterion then determines which variables must be conditioned on for valid causal estimation.
- **Next step**: apply DoWhy to an observational comparison group (e.g., PSID or CPS) where confounding is stronger and the choice of estimator matters more.

## Exercises

1. **Change the number of strata.** Re-run the propensity score stratification with `num_strata=10` and `num_strata=20`. How does the ATE estimate change? What are the tradeoffs of using more vs. fewer strata with a sample of only 445 observations?

2. **Add an additional refutation test.** DoWhy supports a `bootstrap_refuter` that re-estimates the effect on bootstrap samples. Implement this refuter and compare its results to the data subset refuter. Are the conclusions similar?

3. **Estimate effects for subgroups.** Split the dataset by `black` (race indicator) and estimate the ATE separately for each subgroup using DoWhy. Does the job training program have a different effect for Black vs. non-Black participants? What might explain any differences you observe?

## References

1. [DoWhy --- Python Library for Causal Inference (PyWhy)](https://www.pywhy.org/dowhy/)
2. [LaLonde, R. (1986). Evaluating the Econometric Evaluations of Training Programs. American Economic Review, 76(4), 604--620.](https://www.jstor.org/stable/1806062)
3. [Dehejia, R. & Wahba, S. (1999). Causal Effects in Nonexperimental Studies: Reevaluating the Evaluation of Training Programs. JASA, 94(448), 1053--1062.](https://doi.org/10.1080/01621459.1999.10473858)
4. [Sharma, A. & Kiciman, E. (2020). DoWhy: An End-to-End Library for Causal Inference. arXiv:2011.04216.](https://arxiv.org/abs/2011.04216)
5. [Horvitz, D. G. & Thompson, D. J. (1952). A Generalization of Sampling Without Replacement from a Finite Universe. JASA, 47(260), 663--685.](https://doi.org/10.1080/01621459.1952.10483446)
6. [Robins, J. M., Rotnitzky, A. & Zhao, L. P. (1994). Estimation of Regression Coefficients When Some Regressors Are Not Always Observed. JASA, 89(427), 846--866.](https://doi.org/10.1080/01621459.1994.10476818)
7. [Nita, C. J. Causal Inference with Python --- Introduction to DoWhy. Medium.](https://medium.com/@chrisjames.nita/causal-inference-with-python-introduction-to-dowhy-ff5799e48985)